# AeroBank Customer Churn Prediction

**Python · pandas · scikit-learn · Classification**

This portfolio notebook analyses AeroBank customer data and compares Logistic Regression with a Decision Tree classifier to predict whether an account remains **Active** or becomes **Closed**.

> **Data note:** The original `AeroBank_data.csv` file is not included in this public portfolio repository. To rerun the notebook, place the dataset in the same directory as this notebook.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    balanced_accuracy_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

## 1. Load the data

The original project was completed in Google Colab. For the public portfolio version, the file path is relative so the notebook is easier to reuse.

In [ ]:
# Dataset is not included in the public repository.
# Place AeroBank_data.csv in the same directory before rerunning.
file_path = 'AeroBank_data.csv'
df = pd.read_csv(file_path)

df.head()

## 2. Data overview

In [ ]:
print("Shape:", df.shape)
print("\nTarget distribution:")
print(df['status'].value_counts())
print("\nTarget proportions:")
print(df['status'].value_counts(normalize=True))

Shape: (10119, 17)

Target distribution:
status
Active    8495
Closed    1624
Name: count, dtype: int64

Target proportions:
status
Active    0.83951
Closed    0.16049
Name: proportion, dtype: float64

The dataset contains **10,119 customers**. The target is imbalanced: approximately **83.95% Active** and **16.05% Closed**. Because of this, model evaluation uses class-specific metrics and balanced accuracy rather than accuracy alone.

## 3. Data quality and preprocessing

In [ ]:
# Fix unrealistic ages identified during the data-quality review
df.loc[(df['age'] < 0) | (df['age'] > 100), 'age'] = np.nan

# Handle missing values
num_cols = ['age', 'credit_limit', 'utilisation_ratio']
cat_cols = ['gender', 'marital_status', 'annual_income', 'account_type']

df[num_cols] = df[num_cols].fillna(df[num_cols].median())
df[cat_cols] = df[cat_cols].fillna('Unknown')

print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 0

In [ ]:
# Define target and features
X = df.drop(columns=['status', 'rowID'])
y = df['status']

# Encode categorical variables
X_encoded = pd.get_dummies(
    X,
    columns=['gender', 'marital_status', 'account_type', 'annual_income'],
    drop_first=True
)

print("Encoded feature shape:", X_encoded.shape)

Encoded feature shape: (10119, 24)

Categorical variables were one-hot encoded with the first category dropped to avoid redundant dummy columns.

## 4. Train-test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Test set size:", len(y_test))
print(y_test.value_counts())

Test set size: 2024
status
Active    1699
Closed     325
Name: count, dtype: int64

The data was split **80/20** using stratification to preserve the Active/Closed class distribution.

## 5. Logistic Regression

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

log_model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    random_state=42
)

log_model.fit(X_train_scaled, y_train)
y_pred_log = log_model.predict(X_test_scaled)

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_log))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_log))

Confusion Matrix:
[[1251  448]
 [  85  240]]

Classification Report:
              precision    recall  f1-score   support

      Active       0.94      0.74      0.82      1699
      Closed       0.35      0.74      0.47       325

    accuracy                           0.74      2024
   macro avg       0.64      0.74      0.65      2024
weighted avg       0.84      0.74      0.77      2024

## 6. Decision Tree Classifier

In [ ]:
dt_model = DecisionTreeClassifier(
    max_depth=5,
    class_weight='balanced',
    random_state=42
)

dt_model.fit(X_train, y_train)
y_pred_dt = dt_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_dt))
print("Balanced Accuracy:", balanced_accuracy_score(y_test, y_pred_dt))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_dt))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_dt))

Accuracy: 0.900691699604743
Balanced Accuracy: 0.9060107755693394

Confusion Matrix:
[[1526  173]
 [  28  297]]

Classification Report:
              precision    recall  f1-score   support

      Active       0.98      0.90      0.94      1699
      Closed       0.63      0.91      0.75       325

    accuracy                           0.90      2024
   macro avg       0.81      0.91      0.84      2024
weighted avg       0.93      0.90      0.91      2024

## 7. Model comparison

| Metric | Logistic Regression | Decision Tree |
| --- | ---: | ---: |
| Accuracy | 73.67% | **90.07%** |
| Balanced Accuracy | 73.74% | **90.60%** |
| Closed Precision | 34.88% | **63.19%** |
| Closed Recall | 73.85% | **91.38%** |
| Closed F1 | 47.38% | **74.72%** |
| False Positives | 448 | **173** |
| False Negatives | 85 | **28** |

The Decision Tree produced the stronger held-out test performance. It correctly identified **297 of 325 Closed customers** and reduced missed closures from **85 to 28** compared with Logistic Regression.

For a customer-retention use case, reducing false negatives is valuable because these represent customers who close their accounts without being identified as churn risks.

These results come from the original 20% test split and would require additional validation before production use.